In [69]:
"""
FinSight-RAG | S&P 500 Company List with Sectors
Source: Wikipedia S&P 500 constituents table
Stack:  PySpark + pandas (for Wikipedia scraping)
"""

import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, upper, trim, count
from pyspark.sql.types import StructType, StructField, StringType

In [70]:
import os
os.environ["JAVA_HOME"] = "/Library/Java/JavaVirtualMachines/temurin-17.jdk/Contents/Home"

from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName("FinSight-RAG-SP500") \
    .getOrCreate()

print("Spark version:", spark.version)

Spark version: 4.1.2


In [71]:
# ── 2. Scrape S&P 500 table from Wikipedia (pandas handles HTML easily) ───────

import pandas as pd
import urllib.request
import ssl
import certifi
from io import BytesIO

WIKI_URL = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"

headers = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

ssl_context = ssl.create_default_context(cafile=certifi.where())
req = urllib.request.Request(WIKI_URL, headers=headers)

print("Fetching S&P 500 list from Wikipedia...")
with urllib.request.urlopen(req, context=ssl_context) as resp:
    html = resp.read()

# Wrap bytes in BytesIO so pandas/lxml doesn't confuse it for a file path
tables = pd.read_html(BytesIO(html))
sp500_pd = tables[0]

print(f"Raw columns: {list(sp500_pd.columns)}")
print(f"Total companies fetched: {len(sp500_pd)}")
sp500_pd.head()

display(sp500_pd)

Fetching S&P 500 list from Wikipedia...
Raw columns: ['Symbol', 'Security', 'GICS Sector', 'GICS Sub-Industry', 'Headquarters Location', 'Date added', 'CIK', 'Founded']
Total companies fetched: 503


,Symbol,Security,GICS Sector,GICS Sub-Industry,Headquarters Location,Date added,CIK,Founded
0,MMM,3M,Industrials,Industrial Conglomerates,"Saint Paul, Minnesota",1957-03-04,66740,1902
1,AOS,A. O. Smith,Industrials,Building Products,"Milwaukee, Wisconsin",2017-07-26,91142,1916
2,ABT,Abbott Laboratories,Health Care,Health Care Equipment,"North Chicago, Illinois",1957-03-04,1800,1888
3,ABBV,AbbVie,Health Care,Biotechnology,"North Chicago, Illinois",2012-12-31,1551152,2013 (1888)
4,ACN,Accenture,Information Technology,IT Consulting & Other Services,"Dublin, Ireland",2011-07-06,1467373,1989
...,...,...,...,...,...,...,...,...
498,XYL,Xylem Inc.,Industrials,Industrial Machinery & Supplies & Components,"White Plains, New York",2011-11-01,1524472,2011
499,YUM,Yum! Brands,Consumer Discretionary,Restaurants,"Louisville, Kentucky",1997-10-06,1041061,1997
500,ZBRA,Zebra Technologies,Information Technology,Electronic Equipment & Instruments,"Lincolnshire, Illinois",2019-12-23,877212,1969
501,ZBH,Zimmer Biomet,Health Care,Health Care Equipment,"Warsaw, Indiana",2001-08-07,1136869,1927


In [72]:
# ── 3. Normalise column names ─────────────────────────────────────────────────
sp500_pd = sp500_pd.rename(columns={
    "Symbol":               "ticker",
    "Security":             "company_name",
    "GICS Sector":          "sector",
    "GICS Sub-Industry":    "sub_industry",
    "Headquarters Location":"headquarters",
    "Date added":           "date_added",
    "CIK":                  "cik",
    "Founded":              "founded",
})

# Keep only relevant columns
sp500_pd = sp500_pd[["ticker", "company_name", "sector", "sub_industry",
                      "headquarters", "date_added"]]



In [73]:
# ── 4. Convert to Spark DataFrame ─────────────────────────────────────────────
schema = StructType([
    StructField("ticker",        StringType(), True),
    StructField("company_name",  StringType(), True),
    StructField("sector",        StringType(), True),
    StructField("sub_industry",  StringType(), True),
    StructField("headquarters",  StringType(), True),
    StructField("date_added",    StringType(), True),
])

sp500_df = spark.createDataFrame(sp500_pd, schema=schema)



In [74]:
# ── 5. Clean & standardise ────────────────────────────────────────────────────
sp500_df = sp500_df \
    .withColumn("ticker",       upper(trim(col("ticker")))) \
    .withColumn("company_name", trim(col("company_name"))) \
    .withColumn("sector",       trim(col("sector"))) \
    .withColumn("sub_industry", trim(col("sub_industry"))) \
    .dropna(subset=["ticker", "sector"])

print(f"\nTotal companies after cleaning: {sp500_df.count()}")




Total companies after cleaning: 503


In [77]:
# ── 6. Show sample ────────────────────────────────────────────────────────────
print("\n── Sample companies ──")
# sp500_df.show(10, truncate=False)
sp500_df.show(sp500_df.count(), truncate=False)


── Sample companies ──
+------+--------------------------------------+----------------------+-------------------------------------------------------+-------------------------------------------+----------+
|ticker|company_name                          |sector                |sub_industry                                           |headquarters                               |date_added|
+------+--------------------------------------+----------------------+-------------------------------------------------------+-------------------------------------------+----------+
|MMM   |3M                                    |Industrials           |Industrial Conglomerates                               |Saint Paul, Minnesota                      |1957-03-04|
|AOS   |A. O. Smith                           |Industrials           |Building Products                                      |Milwaukee, Wisconsin                       |2017-07-26|
|ABT   |Abbott Laboratories                   |Health Care        

In [118]:
os.makedirs("./storage/raw", exist_ok=True)
sp500_pd.to_json("./storage/raw/sp500_companies.json", orient="records", lines=True)

In [121]:
import datetime
today = datetime.date.today()

'Data fetched and saved on: 2026-06-21'

In [122]:
sp500_df = sp500_df.withColumn("date_added", trim(col("date_added")))

In [128]:
os.makedirs("./storage/raw", exist_ok=True)
sp500_df.write \
    .mode("overwrite") \
    .partitionBy("date_added") \
    .parquet("./storage/raw/sp500_companies.parquet")

Py4JJavaError: An error occurred while calling o514.parquet.
: org.apache.spark.SparkException: [INTERNAL_ERROR] Eagerly executed command failed. You hit a bug in Spark or the Spark plugins you use. Please, report this bug to the corresponding communities or vendors, and provide the full stack trace. SQLSTATE: XX000
	at org.apache.spark.SparkException$.internalError(SparkException.scala:107)
	at org.apache.spark.sql.execution.QueryExecution$.toInternalError(QueryExecution.scala:706)
	at org.apache.spark.sql.execution.QueryExecution$.withInternalError(QueryExecution.scala:719)
	at org.apache.spark.sql.execution.QueryExecution.org$apache$spark$sql$execution$QueryExecution$$eagerlyExecute$1(QueryExecution.scala:184)
	at org.apache.spark.sql.execution.QueryExecution$$anonfun$eagerlyExecuteCommands$3.applyOrElse(QueryExecution.scala:201)
	at org.apache.spark.sql.execution.QueryExecution$$anonfun$eagerlyExecuteCommands$3.applyOrElse(QueryExecution.scala:194)
	at org.apache.spark.sql.catalyst

In [78]:
# ── 7. Sector summary ─────────────────────────────────────────────────────────
print("\n── Companies per sector ──")
sector_summary = sp500_df \
    .groupBy("sector") \
    .agg(count("ticker").alias("company_count")) \
    .orderBy(col("company_count").desc())

sector_summary.show(truncate=False)




── Companies per sector ──
+----------------------+-------------+
|sector                |company_count|
+----------------------+-------------+
|Industrials           |80           |
|Financials            |76           |
|Information Technology|74           |
|Health Care           |59           |
|Consumer Discretionary|47           |
|Consumer Staples      |35           |
|Real Estate           |31           |
|Utilities             |31           |
|Materials             |26           |
|Communication Services|23           |
|Energy                |21           |
+----------------------+-------------+



In [79]:
# ── 8. Filter by sector (example: Technology) ─────────────────────────────────
TARGET_SECTOR = "Information Technology"

tech_companies = sp500_df \
    .filter(col("sector") == TARGET_SECTOR) \
    .select("ticker", "company_name", "sub_industry") \
    .orderBy("ticker")

print(f"\n── {TARGET_SECTOR} companies ──")
# tech_companies.show(20, truncate=False)
tech_companies.show(tech_companies.count(), truncate=False)


── Information Technology companies ──
+------+--------------------------+------------------------------------------+
|ticker|company_name              |sub_industry                              |
+------+--------------------------+------------------------------------------+
|AAPL  |Apple Inc.                |Technology Hardware, Storage & Peripherals|
|ACN   |Accenture                 |IT Consulting & Other Services            |
|ADBE  |Adobe Inc.                |Application Software                      |
|ADI   |Analog Devices            |Semiconductors                            |
|ADSK  |Autodesk                  |Application Software                      |
|AKAM  |Akamai Technologies       |Internet Services & Infrastructure        |
|AMAT  |Applied Materials         |Semiconductor Materials & Equipment       |
|AMD   |Advanced Micro Devices    |Semiconductors                            |
|ANET  |Arista Networks           |Communications Equipment                  |
|APH   |Amph

In [45]:
# ── 9. Save to PostgreSQL (FinSight-RAG pipeline) ─────────────────────────────
# Uncomment and configure when running in Databricks with a JDBC connection

# JDBC_URL = "jdbc:postgresql://<host>:5432/<dbname>"
# DB_PROPS = {
#     "user":   dbutils.secrets.get(scope="finsight", key="pg-user"),
#     "password": dbutils.secrets.get(scope="finsight", key="pg-password"),
#     "driver": "org.postgresql.Driver"
# }
#
# sp500_df.write \
#     .jdbc(url=JDBC_URL, table="sp500_companies", mode="overwrite",
#           properties=DB_PROPS)
# print("Saved to PostgreSQL: sp500_companies")



In [80]:
# ── 10. Return ticker list (for downstream yfinance calls) ────────────────────
ticker_list = [row["ticker"] for row in sp500_df.select("ticker").collect()]
print(f"\nTicker list ready: {ticker_list[:10]} ... ({len(ticker_list)} total)")

spark.stop()


Ticker list ready: ['MMM', 'AOS', 'ABT', 'ABBV', 'ACN', 'ADBE', 'AMD', 'AES', 'AFL', 'A'] ... (503 total)


In [93]:
def get_stock_names():
    print("Fetching S&P 500 stocks...")
    sp500_df = sp500_df.show(sp500_df.count(), truncate=False)
    return sp500_df